In [ ]:
import numpy as np
from ipywidgets import FloatSlider, Dropdown, HBox, Layout, VBox, HTML
from IPython.display import display


def ieee754_bits(value, precision):
    if precision == 'Half precision (16 bit)':
        stored = np.float16(value)
        raw = np.array([stored], dtype=np.float16).view(np.uint16)[0]
        bits = format(int(raw), '016b')
        exponent_bits = 5
        fraction_bits = 10
        bias = 15

    elif precision == 'Single precision (32 bit)':
        stored = np.float32(value)
        raw = np.array([stored], dtype=np.float32).view(np.uint32)[0]
        bits = format(int(raw), '032b')
        exponent_bits = 8
        fraction_bits = 23
        bias = 127

    else:
        stored = np.float64(value)
        raw = np.array([stored], dtype=np.float64).view(np.uint64)[0]
        bits = format(int(raw), '064b')
        exponent_bits = 11
        fraction_bits = 52
        bias = 1023

    sign = bits[0]
    exponent = bits[1:1 + exponent_bits]
    fraction = bits[1 + exponent_bits:]

    return stored, sign, exponent, fraction, exponent_bits, fraction_bits, bias


summary_html = HTML()
sign_html = HTML()
details_html = HTML()


slider_layout = Layout(width='300px')

style_opts = {'description_width': '70px'}


value_slider = FloatSlider(
    min=-100.0,
    max=100.0,
    step=0.1,
    value=10.3,
    description='Value:',
    style=style_opts,
    layout=slider_layout,
    continuous_update=True
)


precision_selector = Dropdown(
    options=[
        'Half precision (16 bit)',
        'Single precision (32 bit)',
        'Double precision (64 bit)'
    ],
    value='Single precision (32 bit)',
    description='Precision:',
    style=style_opts,
    layout=slider_layout
)


def update_display(*args):
    value = value_slider.value
    precision = precision_selector.value

    stored, sign, exponent, fraction, exponent_bits, fraction_bits, bias = ieee754_bits(value, precision)

    error = float(value) - float(stored)

    exponent_decimal = int(exponent, 2)
    actual_exponent = exponent_decimal - bias

    if precision == 'Half precision (16 bit)':
        total_bits = 16

    elif precision == 'Single precision (32 bit)':
        total_bits = 32

    else:
        total_bits = 64

    summary_html.value = f"""
    <div style="
        font-family: monospace;
        font-size: 14px;
        line-height: 1.75;
        white-space: nowrap;
    ">

    <b>Selected precision:</b> {precision}<br>
    <b>Total word length:</b> {total_bits} bits<br>
    <b>Original value:</b> {value:.17g}<br>
    <b>Stored value:</b> {float(stored):.17g}<br>
    <b>Representation error:</b> {error:.17g}

    </div>
    """

    sign_html.value = f"""
    <div style="
        font-family: monospace;
        font-size: 14px;
        line-height: 1.75;
    ">

    <b>Sign bit:</b><br>
    <span style="font-size: 18px;">{sign}</span>

    </div>
    """

    details_html.value = f"""
    <div style="
        font-family: monospace;
        font-size: 14px;
        line-height: 1.75;
        margin-top: 18px;
    ">

    <b>Exponent ({exponent_bits} bits):</b><br>
    <span style="font-size: 18px;">{exponent}</span><br>

    Stored exponent = {exponent_decimal}<br>
    Bias = {bias}<br>
    Actual exponent = {actual_exponent}<br><br>

    <b>Fraction ({fraction_bits} bits):</b><br>
    <span style="
        font-size: 16px;
        word-break: break-all;
        display: inline-block;
        max-width: 700px;
    ">
    {fraction}
    </span>

    </div>
    """


value_slider.observe(update_display, names='value')

precision_selector.observe(update_display, names='value')


theory_html = HTML("""
<div style="
    font-family: monospace;
    font-size: 13px;
    line-height: 1.55;
    margin-bottom: 12px;
">

<b>Half precision:</b> 1 sign bit, 5 exponent bits, 10 fraction bits.<br>
<b>Single precision:</b> 1 sign bit, 8 exponent bits, 23 fraction bits.<br>
<b>Double precision:</b> 1 sign bit, 11 exponent bits, 52 fraction bits.<br><br>

<b>Value:</b> Selects the decimal value to be encoded using the IEEE 754 representation.<br>
Changing the value modifies the sign, exponent, and fraction fields according to the selected precision.<br><br>

<b>Precision:</b> Selects half, single, or double precision.<br>
Increasing the precision increases the number of exponent and fraction bits, expands the available dynamic range, and improves the accuracy of the stored value.<br>
For values that cannot be represented exactly, the representation error generally decreases as the precision increases.<br><br>

The notebook displays the actual IEEE 754 bit pattern used for the selected numerical value.

</div>
""")


title_html = HTML("""
<div style="
    font-family: monospace;
    font-size: 14px;
    margin-bottom: 12px;
">
<b>IEEE 754 representation</b>
</div>
""")


summary_html.layout = Layout(
    width='500px',
    min_width='500px'
)


sign_html.layout = Layout(
    width='160px',
    min_width='160px'
)


controls = VBox(
    [value_slider, precision_selector],
    layout=Layout(
        width='310px',
        min_width='310px',
        overflow='visible',
        justify_content='flex-start',
        margin='0 0 0 10px'
    )
)


sign_controls_row = HBox(
    [sign_html, controls],
    layout=Layout(
        width='500px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible',
        margin='18px 0 0 0'
    )
)


main_content = VBox(
    [title_html, summary_html, sign_controls_row, details_html],
    layout=Layout(
        width='850px',
        align_items='flex-start',
        overflow='visible'
    )
)


update_display()


display(
    VBox(
        [theory_html, main_content],
        layout=Layout(
            width='100%',
            align_items='flex-start',
            overflow='visible'
        )
    )
)